# Lab 04: How to find the stage where a RAG answer failed

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | In class**

## Goal

Follow every question through the pipeline, find the wrong answers, and name the stage where each one went wrong. The
moment that teaches it: the three wrong answers all failed at the same stage, generation, after the right passage had
been found and packed every time, so **the fix is not in the search**. Part E (self-study) then gives the model that
passage alone and tells the three causes apart: one answer misled by the other passages, one right in words the answer
key did not list, one the model still refuses.

## What you will learn

- What a trace record holds, and how to make one by hand.
- How to trace a question set and read its cost from the records.
- How to screen answers against expected strings, and read what the screen flags.
- How to build a stage table that follows the support through the hits, the pack and the answer.
- How to use oracle context to separate a failure caused by the other evidence from one caused by the answer step,
  and to spot an incomplete label.
- How to save a baseline run that later labs compare against.

**In class:** Parts A and D; run the cells of Parts B and C without stopping on them, because Part D reads what they
make. Parts B, C, E and F are self-study.


## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-26
OPENAI_API_KEY    found in .env


About 90 short chat calls on `chat_model`, well under one US cent on `gpt-4o-mini`. Every count in the text comes from
`gpt-4o-mini`; on another model the failures can move to other questions. Replies vary from run to run, so compare the kind of reply with
the text, not every number.


---

## Part A: Trace one question by hand

Part A records what each step of the pipeline produced for one question, so that a wrong answer can be read step by
step afterwards. That record is a **trace**; tracing tools such as OpenTelemetry call the record of one step a
**span**, and keeping such records is **observability**.

The pipeline is Lab 02's: search the 272 chunks and the five policies, pack the best hits as `[ID] text` lines within
400 tokens, ask for a cited answer as JSON. The trace keeps one thing per step:

| Step | The trace keeps | It answers |
|---|---|---|
| Search | the hits: chunk IDs and scores | Was the right passage found, and how high? |
| Packing | the packed IDs | Did it reach the prompt? |
| Prompt | a short fingerprint of the messages (Part B) | Was this the same prompt as last time? |
| Answer | the raw reply and the tokens billed | What did the model say, and what did it cost? |
| Run | the stamp | Which models, which endpoint, which day? |

Start with the corpus and its cached vectors. The imports come from earlier labs:

- `chunk_vectors`, `search` (Lab 01): the cached chunk vectors and the top-k search.
- `pack` (Lab 02 Part A): the `[ID] text` lines within a token budget.
- `parse`, `words` (Lab 02 Part E): the JSON shape check; a text as lower-case words.
- `RULES`, `response_format` (Lab 02 Part G): the system message and the JSON schema the reply must follow.

The model answers as JSON: an `answer` and a list of `claims`, each with a quote and a source ID. Lab 02 built this;
here it is read, not built.


In [3]:
# corpus: the 272 chunks plus the five policies, each policy as a chunk whose ID is its source ID
# vectors: one embedding per record, from the cache Lab 01 built
import json
from shared.search import chunk_vectors, search
from shared.answer import RULES, pack, parse, response_format, words

DATA = ROOT / "data"
chunks = json.loads((DATA / "squad-chunks.json").read_text(encoding="utf-8"))
policies = json.loads((DATA / "support-case.json").read_text(encoding="utf-8"))
corpus = chunks + [{**policy, "chunk_id": policy["source_id"]} for policy in policies]
vectors = chunk_vectors(corpus, settings)
print(len(corpus), "chunks |", vectors.shape)


277 chunks | (277, 1536)


277 records, one vector each. Search and pack the Tesla question:


In [4]:
# hits: the five best (score, chunk) pairs for the question; lines: the "[ID] text" lines that fit in 400 tokens
question = "In what year was Nikola Tesla born?"
hits = search(question, corpus, vectors, settings, k=5)
lines = pack(hits, settings)
print("hits:  ", [(chunk["chunk_id"], round(score, 4)) for score, chunk in hits])
print("packed:", [line[1:line.index("]")] for line in lines])


hits:   [('S03-P00-C00', 0.7222), ('S03-P01-C00', 0.5405), ('S03-P02-C00', 0.4657), ('S03-P01-C01', 0.2591), ('S06-P00-C00', 0.25)]
packed: ['S03-P00-C00', 'S03-P01-C00', 'S03-P02-C00', 'S03-P01-C01']


- `hits`: five (ID, score) pairs, best first; the score says how close a chunk came, not only whether it was packed.
- `packed`: the four IDs that fit the 400-token budget; the fifth hit would have passed it.

Next, the prompt as `answer()` builds it: the system message holds the rules, with the line that allows *Insufficient
evidence.*; the user message holds the question and the evidence lines. Print both before they are sent:


In [5]:
# user: the question and the evidence lines as one text; messages: the system and user messages the call sends
user = f"Question: {question}\nEvidence:\n" + "\n".join(lines)
messages = [{"role": "system", "content": RULES}, {"role": "user", "content": user}]
print("system:", RULES)
print("\nuser:", user[:250], "...")


system: Answer the question in one sentence, using only the evidence. List each claim the answer makes, with the ID of the evidence line it comes from and a quote: words copied exactly from that line. If the evidence does not answer the question, the answer is "Insufficient evidence." and the claims list is empty. Reply in JSON shaped like {"answer": "...", "claims": [{"claim": "...", "source_id": "...", "quote": "..."}]}.

user: Question: In what year was Nikola Tesla born?
Evidence:
[S03-P00-C00] Nikola Tesla (Serbian Cyrillic: Никола Тесла; 10 July 1856 – 7 January 1943) was a Serbian American inventor, electrical engineer, mechanical engineer, physicist, and futurist best ...


Then the call `answer()` makes, except that the reply stays text, unparsed: if a reply is broken, the trace should
show how.

- `chat_options("CHAT_MODEL", settings)`: the model and its options from `models.env`.
- `response_format(settings)`: the JSON schema.
- `reply.usage`: the tokens billed, in and out.


In [6]:
# reply: the API's response; .choices[0].message.content is the reply text, .usage the tokens billed
from shared.settings import chat_client, chat_options, chat_url

reply = chat_client(settings).chat.completions.create(
    **chat_options("CHAT_MODEL", settings), response_format=response_format(settings), messages=messages)
print(reply.choices[0].message.content)
print("tokens billed:", reply.usage.prompt_tokens, "in,", reply.usage.completion_tokens, "out")


{"answer":"Nikola Tesla was born on 10 July 1856.","claims":[{"claim":"Nikola Tesla was born on 10 July 1856.","source_id":"S03-P00-C00","quote":"10 July 1856"}]}
tokens billed: 500 in, 53 out


One dictionary holds the pieces, with the stamp from Config. This is the trace record:


In [7]:
# record: the trace of one question, one field per step
record = {
    "question": question,
    "hits": [[chunk["chunk_id"], round(score, 4)] for score, chunk in hits],
    "packed": [line[1:line.index("]")] for line in lines],
    "reply": reply.choices[0].message.content,
    "usage": {"input": reply.usage.prompt_tokens, "output": reply.usage.completion_tokens},
    "stamp": stamp(settings),
}
for name, value in record.items():
    print(f"{name:9}{value}")


question In what year was Nikola Tesla born?
hits     [['S03-P00-C00', 0.7222], ['S03-P01-C00', 0.5405], ['S03-P02-C00', 0.4657], ['S03-P01-C01', 0.2591], ['S06-P00-C00', 0.25]]
packed   ['S03-P00-C00', 'S03-P01-C00', 'S03-P02-C00', 'S03-P01-C01']
reply    {"answer":"Nikola Tesla was born on 10 July 1856.","claims":[{"claim":"Nikola Tesla was born on 10 July 1856.","source_id":"S03-P00-C00","quote":"10 July 1856"}]}
usage    {'input': 500, 'output': 53}
stamp    {'chat_model': 'gpt-4o-mini', 'reasoning_effort': None, 'embed_model': 'text-embedding-3-small', 'judge_model': 'gpt-4o-mini', 'rerank_model': 'gpt-4o-mini', 'chat_endpoint': 'https://api.openai.com/v1', 'embed_endpoint': 'https://api.openai.com/v1', 'run_date': '2026-10-26'}


- `question`: the text as asked.
- `hits`, `packed`: the two lists above.
- `reply`: the raw JSON text, parsed later or read as it came.
- `usage`: about 500 input tokens, more than Lab 02's plain call because the rules and the schema are in the prompt.
- `stamp`: the models, the endpoint and the day.

> A trace keeps what each stage produced, not only the answer. **A wrong answer can then be read stage by stage.**

### Checkpoint A

| You should see | Our run |
|---|---|
| Corpus | 277 chunks, vectors `(277, 1536)` |
| Tesla's trace | 5 hits, `S03-P00-C00` first (about 0.72); 4 packed |
| The reply | JSON with *Nikola Tesla was born on 10 July 1856.* (or *in the year 1856*) and one claim from `S03-P00-C00`; 500 input tokens |


---

## Part B: Trace every question (self-study)

Part B adds the prompt's fingerprint, turns Part A into a function and runs it on every question, so that each wrong
answer comes with its record.

A **hash** is a short fingerprint of a text: the same text always gives the same hash, and any change, even one
character, gives a different one. Two traces with the same hash had the same prompt, word for word. Hash Part A's
messages:


In [8]:
# prompt_sha: the first 12 hex characters of the SHA-256 hash of Part A's messages
import hashlib

prompt_sha = hashlib.sha256(json.dumps(messages).encode()).hexdigest()[:12]
print(prompt_sha)


c2aa8c003db7


### What happens step by step

1. `json.dumps(messages)`: the two messages as one string.
2. `.encode()`: the string as bytes, what a hash function reads.
3. `hashlib.sha256(...)`: the fingerprint, 64 hex characters.
4. `.hexdigest()[:12]`: the first 12, enough to tell prompts apart here.

`trace()` is Part A's cells joined, with the hash as `prompt_sha`. Its hash for the Tesla question must match the one
above:


In [9]:
# trace(): search, pack, build the messages, call the model once, and return one record with the prompt hash
def trace(question, k=5):
    hits = search(question, corpus, vectors, settings, k=k)
    lines = pack(hits, settings)
    user = f"Question: {question}\nEvidence:\n" + "\n".join(lines)
    messages = [{"role": "system", "content": RULES}, {"role": "user", "content": user}]
    reply = chat_client(settings).chat.completions.create(
        **chat_options("CHAT_MODEL", settings), response_format=response_format(settings), messages=messages)
    return {"question": question,
            "hits": [[chunk["chunk_id"], round(score, 4)] for score, chunk in hits],
            "packed": [line[1:line.index("]")] for line in lines],
            "prompt_sha": hashlib.sha256(json.dumps(messages).encode()).hexdigest()[:12],
            "reply": reply.choices[0].message.content,
            "usage": {"input": reply.usage.prompt_tokens, "output": reply.usage.completion_tokens},
            "stamp": stamp(settings)}

print("same prompt as Part A:", trace(question)["prompt_sha"] == prompt_sha)


same prompt as Part A: True


### What happens step by step

1. `search()` ranks the corpus; `pack()` keeps the lines that fit.
2. The messages are built as in Part A.
3. One chat call with the schema; the reply stays text.
4. The record is returned, `prompt_sha` included.

Now the questions. Each of the 30 development questions comes with SQuAD's answer strings and `known_support`, its
paragraph. Look at the first one's `answers` field:


In [10]:
# dev: the 30 development questions as the file holds them; the last line shows the first question's answers field
dev = json.loads((DATA / "development-questions.json").read_text(encoding="utf-8"))
print(dev[0]["question"], "| known support:", dev[0]["known_support"])
dev[0]["answers"]


Which NFL team represented the AFC at Super Bowl 50? | known support: ['S00-P00']


[{'answer_start': 177, 'text': 'Denver Broncos'},
 {'answer_start': 177, 'text': 'Denver Broncos'},
 {'answer_start': 177, 'text': 'Denver Broncos'}]

`answers` holds one dictionary per person who marked the answer: the marked `text` and where it starts. Three people
marked the same words, so `sorted({a["text"] for a in q["answers"]})` keeps each wording once, in a fixed order.

Give every question three fields, `question`, `expected` and `support`, and add four Atlas questions written for this
lab, with the answers the policies give:


In [11]:
# questions: 34 records with question, expected (the answer strings) and support (the paragraph or policy IDs)
questions = [{"question": q["question"], "expected": sorted({a["text"] for a in q["answers"]}),
              "support": q["known_support"]} for q in dev]
questions += [  # written for this lab: the answers the Atlas policies give
    {"question": "How long does an Atlas reset link remain valid?", "expected": ["30 minutes"], "support": ["H02"]},
    {"question": "Who can export all reports in Atlas?", "expected": ["workspace owners"], "support": ["H03"]},
    {"question": "Can temporary Atlas accounts reset by email?", "expected": ["support agent"], "support": ["H01"]},
    {"question": "Do I need the mobile app to reset my Atlas password?", "expected": ["no", "by email"], "support": ["H01"]},
]
print(len(questions), "questions")


34 questions


Trace all of them, one call each, and add up the tokens from the records:


In [12]:
# traces: one record per question; the token sums come from the records' usage fields
traces = [trace(q["question"]) for q in questions]
print(len(traces), "traces |", sum(t["usage"]["input"] for t in traces), "input tokens,",
      sum(t["usage"]["output"] for t in traces), "output tokens")


34 traces | 17641 input tokens, 2357 output tokens


About 17,600 input and 2,400 output tokens, added up from the records. **A trace is also the bill.**

### Checkpoint B

| You should see | Our run |
|---|---|
| The hash | `c2aa8c003db7`; `same prompt as Part A: True` |
| All questions | 34 traces; 17,641 input tokens; output tokens vary by run (2,357 in ours) |


---

## Part C: Screen the answers (self-study)

Part C picks out the traces worth reading. A **screen** is a quick check in code that flags answers for a person to
read; it does not grade them. This one is an **exact-match screen**:

$$
\text{pass}(a, E) = \text{some } e \in E \text{ appears in } a \text{ as whole words}
$$

- $a$: the answer as lower-case words (Lab 02's `words()`).
- $E$: the question's `expected` strings.
- whole words: a space at each end of both texts, so that *no* is not found inside *know*.

Define it and try it on the first trace:


In [13]:
# screen(): True when one expected string appears in the answer as whole words; tried on the first trace
def screen(answer_text, expected):
    answer = f" {words(answer_text)} "
    return any(f" {words(text)} " in answer for text in expected)

first = parse(traces[0]["reply"])["answer"]
print(first, "| expected:", questions[0]["expected"])
print("passes:", screen(first, questions[0]["expected"]))


The Denver Broncos represented the AFC at Super Bowl 50. | expected: ['Denver Broncos']
passes: True


`parse(t["reply"])["answer"]` turns the raw JSON back into a dictionary and takes the answer sentence. Now all 34,
printing the ones that fail:


In [14]:
# answers: the answer sentence of every trace; passed: one True or False per question, in the same order
answers = [parse(t["reply"])["answer"] for t in traces]
passed = [screen(a, q["expected"]) for a, q in zip(answers, questions)]
print(f"passed {sum(passed)} of {len(passed)}")
for a, q, ok in zip(answers, questions, passed):
    if not ok:
        print(f"\n{q['question']}\n  expected: {q['expected']}\n  answer:   {a}")


passed 31 of 34

What measure of a computational problem broadly defines the inherent difficulty of the solution?
  expected: ['if its solution requires significant resources', 'its solution requires significant resources']
  answer:   Computational complexity theory broadly defines the inherent difficulty of a computational problem.

Can temporary Atlas accounts reset by email?
  expected: ['support agent']
  answer:   Insufficient evidence.

Do I need the mobile app to reset my Atlas password?
  expected: ['no', 'by email']
  answer:   Yes, password reset uses a mobile app.


About 31 of 34 pass; Part D reads the flagged answers. A pass is not proof: the screen checks that the words
are there, not that the evidence supports them (Lab 02's quote check and Lab 07's judge read more). **The screen
decides what to read first.**

### Checkpoint C

| You should see | Our run |
|---|---|
| The screen | `passed 31 of 34` |
| Flagged | the complexity question, the temporary-accounts question and the mobile-app question |


---

## Part D: Find the stage where the support was lost

Part B traced 34 questions, the 30 development questions plus four Atlas questions written in Part B, each with its
`support` (the paragraph or policy that holds the answer) and its `expected` answer strings, into `traces`, one record
each like Part A's; `questions` holds the 34 in the same order. Part C screened each answer for its expected strings
into `passed`, True or False per question, and kept the answer sentences in `answers`.

Part D asks where each flagged answer lost its **support**, the passage that holds the answer: the `known_support`
paragraph, or the policy for the Atlas questions. The method is **failure attribution by stage**: follow the support
through the pipeline in order; the first stage it does not reach is the one to inspect.

Start with the answers the screen flagged, beside their support and expected strings:


In [15]:
# flagged: the questions whose answer failed the screen, with their support, expected strings and the answer given
flagged = [(q, a) for q, a, ok in zip(questions, answers, passed) if not ok]
for q, a in flagged:
    print(f"{q['question']}\n  support: {q['support']} | expected: {q['expected']}\n  answer:  {a}\n")


What measure of a computational problem broadly defines the inherent difficulty of the solution?
  support: ['S04-P01'] | expected: ['if its solution requires significant resources', 'its solution requires significant resources']
  answer:  Computational complexity theory broadly defines the inherent difficulty of a computational problem.

Can temporary Atlas accounts reset by email?
  support: ['H01'] | expected: ['support agent']
  answer:  Insufficient evidence.

Do I need the mobile app to reset my Atlas password?
  support: ['H01'] | expected: ['no', 'by email']
  answer:  Yes, password reset uses a mobile app.



Three wrong answers, three different ways:

- The complexity question gets the name of the field, *computational complexity theory*; the expected strings
  name what makes a problem hard, *its solution requires significant resources*.
- The temporary-accounts question is refused, though its support H01 says temporary accounts *require a support
  agent*.
- The mobile-app question gets *Yes, password reset uses a mobile app.* That sentence is in H04, the Boreal policy,
  not in its support H01.

Each stage can lose the support; the first it does not reach names where to look:

| The support is ... | Stage | What to inspect next |
|---|---|---|
| not in the corpus | corpus | whether the document was loaded at all |
| in the corpus, not in the hits | retrieval | the chunks, the question's wording, the ranking |
| in the hits, not in the pack | packing | the token budget, the order, any filter |
| in the pack, and the answer is wrong | generation | the rules and what the model did with the evidence |

> *What measure of a computational problem broadly defines the inherent difficulty of the solution?*
>
> *Can temporary Atlas accounts reset by email?*
>
> *Do I need the mobile app to reset my Atlas password?*

*Predict first:* for each, was the right passage missing from the corpus, from the hits, from the pack, or did the
model have it and still fail?

The hits and the pack hold chunk IDs (`S04-P01-C00`); the support is a paragraph ID (`S04-P01`), so map each chunk ID
to its paragraph first:


In [16]:
# source_of: chunk ID -> paragraph ID; stages(): True or False for each stage one support reached; table: one row per question
source_of = {chunk["chunk_id"]: chunk["source_id"] for chunk in corpus}

def stages(t, q, ok):
    return {"corpus": any(chunk["source_id"] in q["support"] for chunk in corpus),
            "hits": any(source_of[chunk_id] in q["support"] for chunk_id, _ in t["hits"]),
            "pack": any(source_of[chunk_id] in q["support"] for chunk_id in t["packed"]),
            "answer": ok}

table = [stages(t, q, ok) for t, q, ok in zip(traces, questions, passed)]
for step in ("corpus", "hits", "pack"):
    print(f"support in the {step:7} {sum(row[step] for row in table)} of {len(table)}")
print(f"answer passes screen {sum(row['answer'] for row in table)} of {len(table)}")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")


support in the corpus  34 of 34
support in the hits    34 of 34
support in the pack    34 of 34
answer passes screen 31 of 34

gpt-4o-mini at https://api.openai.com/v1


### What happens step by step

1. `source_of` maps every chunk ID to its paragraph ID; a policy maps to itself.
2. `stages()` answers four yes/no questions for one trace: support in the corpus, in the hits, in the pack, and did
   the answer pass the screen.
3. `table` holds one row per question; the loop counts, per stage, how many supports reached it.

The support reached the corpus, the hits and the pack for all 34 questions; nothing was lost before the
answer step. Name the stage for each flagged trace and print where its support ranked. `next((step for step in row if
not row[step]), None)` returns the first stage marked False, or `None` when all four are True:


In [17]:
# lost: the first stage whose value is False, None when every stage passed; ranks: the hit positions of the support
# the line also shows the packed IDs, so the pack can be read beside the ranks
STAGE = {"corpus": "corpus", "hits": "retrieval", "pack": "packing", "answer": "generation"}

for t, q, row in zip(traces, questions, table):
    lost = next((step for step in row if not row[step]), None)
    if lost:
        ranks = [i + 1 for i, (chunk_id, _) in enumerate(t["hits"]) if source_of[chunk_id] in q["support"]]
        print(f"{STAGE[lost]:10} | support at hit {ranks} | packed {t['packed']} | {q['question']}")


generation | support at hit [1, 4] | packed ['S04-P01-C00', 'S04-P00-C00', 'S04-P02-C01', 'S04-P01-C01', 'S04-P02-C00'] | What measure of a computational problem broadly defines the inherent difficulty of the solution?
generation | support at hit [1] | packed ['H01', 'H04', 'H02', 'H05', 'H03'] | Can temporary Atlas accounts reset by email?
generation | support at hit [2] | packed ['H04', 'H01', 'H02', 'H05', 'H03'] | Do I need the mobile app to reset my Atlas password?


All three stop at `generation`: the support was packed, and for two of them it was the first hit. On the
mobile-app question H04, the Boreal policy, ranked first and its support H01 second. The complexity paragraph
`S04-P01` is in the pack twice, as its chunks `S04-P01-C00` and `S04-P01-C01`, at hits 1 and 4.

> The stage table says where the support stopped. It cannot say why the model failed with it: that needs oracle
> context (Part E).

### Checkpoint D

| You should see | Our run |
|---|---|
| Support | corpus, hits and pack 34 of 34 each; `answer passes screen 31 of 34` |
| Stages | `generation` for all three; support at hits 1 and 4 (complexity), 1 (temporary accounts), 2 (mobile app) |


---

## Part E: Give the model only the labelled passage (self-study)

Part E tests the answer step on its own. **Oracle context** replaces the retrieved evidence with the passage the label
names, so that search and packing cannot be the cause. It is a diagnostic only: a real question comes with no label,
so an oracle run is never a result to report.

- The answer becomes right: the model can answer from the support; the trouble is in what else was packed next to it.
- The answer stays wrong: the trouble is in the answer step, the rules or the model.

Collect the passage texts and the flagged questions:


In [18]:
# passages: paragraph or policy ID -> its full text; failed: the questions the screen flagged
passages = {p["source_id"]: p["text"] for p in json.loads((DATA / "squad-passages.json").read_text(encoding="utf-8"))}
passages.update({p["source_id"]: p["text"] for p in policies})

failed = [q for q, ok in zip(questions, passed) if not ok]
for q in failed:
    print(q["support"], "|", passages[q["support"][0]][:110], "...")


['S04-P01'] | A problem is regarded as inherently difficult if its solution requires significant resources, whatever the alg ...
['H01'] | Atlas passwords can be reset by email. Temporary accounts require a support agent. ...
['H01'] | Atlas passwords can be reset by email. Temporary accounts require a support agent. ...


Ask each flagged question five times with only its support as evidence, through `answer()` from `shared/answer.py`
(the same rules and schema), and screen every answer:


In [19]:
# oracle: the support passage as the only evidence line; oracle_answers: five answers from it, counted by wording
from collections import Counter
from shared.answer import answer

for q in failed:
    oracle = [f"[{source_id}] {passages[source_id]}" for source_id in q["support"]]
    oracle_answers = [answer(q["question"], oracle, settings)["answer"] for _ in range(5)]
    print(f"\n{q['question']} | passes the screen {sum(screen(a, q['expected']) for a in oracle_answers)} of 5")
    for text, n in Counter(oracle_answers).most_common():
        print(f"  {n} of 5 | {text}")



What measure of a computational problem broadly defines the inherent difficulty of the solution? | passes the screen 0 of 5
  5 of 5 | The measure of a computational problem that broadly defines the inherent difficulty of the solution is the amount of resources needed to solve them, such as time and storage.



Can temporary Atlas accounts reset by email? | passes the screen 0 of 5
  5 of 5 | Insufficient evidence.



Do I need the mobile app to reset my Atlas password? | passes the screen 5 of 5
  5 of 5 | No, you do not need the mobile app to reset your Atlas password.


### What happens step by step

1. `oracle`: one `[ID] text` line per support passage, in place of the packed hits.
2. `answer()` asks five times from that line alone.
3. `Counter(...).most_common()` groups the five answers by wording, most frequent first.

Three results:

- Mobile app, from H01 alone: *No, you do not need the mobile app to reset your Atlas password.* in 5 of 5, and the
  screen passes it. The model can answer from H01, so another packed line misled it: H04's second sentence, *Password
  reset uses a mobile app.*, names no product, and H04 ranked first.
- Complexity: *the amount of resources needed to solve them, such as time and storage* in 5 of 5, and the
  screen fails all five. The passage says the theory works by *quantifying the amount of resources needed to solve
  them*: the answer is right, in other words than the label's. SQuAD's label lists the words people marked, not every
  right wording, so **the label is incomplete** and the screen calls a right answer a miss. In the pipeline, the other
  S04 chunks, about classifying problems by *inherent difficulty*, pulled the answer to the name of the field.
- Temporary accounts: still *Insufficient evidence.* in 5 of 5, with only the passage that holds the answer. Search
  and packing are not the cause. Take the abstention line out of the rules and ask again from H01 alone:


In [20]:
# rules_without: the rules with the abstention line removed; oracle_answers: five answers from H01 alone under them
ABSTAIN = ' If the evidence does not answer the question, the answer is "Insufficient evidence." and the claims list is empty.'
rules_without = RULES.replace(ABSTAIN, "")
temporary = next(q for q in questions if "temporary" in q["question"])

print("abstention line removed:", rules_without != RULES)
oracle_answers = [answer(temporary["question"], [f"[H01] {passages['H01']}"], settings, rules=rules_without)["answer"]
                  for _ in range(5)]
for text, n in Counter(oracle_answers).most_common():
    print(f"  {n} of 5 | {text}")


abstention line removed: True


  5 of 5 | Temporary Atlas accounts cannot reset by email; they require a support agent.


Without the line, 5 of 5 answers say temporary accounts cannot reset by email and require a support agent. **The
abstention line causes this refusal**: H01 states the rule, the answer takes one step of reasoning from it, and with
the line the model refuses to take that step.

| Question | Stage table | Oracle context | Where to look next |
|---|---|---|---|
| *Do I need the mobile app to reset my Atlas password?* | generation | right in 5 of 5 | what was packed beside the support: H04, another product's policy, ranked first. Keep other products' policies out of the pack, for example with a metadata filter (Lab 11) |
| *What measure of a computational problem broadly defines ...?* | generation | right in other words in 5 of 5; the screen fails it | the label: accept other wordings, or have a person or a checked judge read it (Lab 07); then the pack, whose other chunks pulled the answer away |
| *Can temporary Atlas accounts reset by email?* | generation | still refused in 5 of 5; answered without the abstention line | the rules: the abstention line refuses an answer that takes one step of reasoning |


**Lesson to Remember**
> Same stage, three repairs: filter the pack, fix the label, change the rule. Changing the rules would not remove H04
> from the pack, and a better search would not change what the abstention line does. Change the stage the evidence
> points to, rerun the same questions, compare the traces.

### Checkpoint E

| You should see | Our run |
|---|---|
| Mobile app, oracle | *No, you do not need the mobile app ...* in 5 of 5; the screen passes it |
| Complexity, oracle | answers about *the amount of resources needed to solve them* in 4 or 5 of 5 (5 in ours); the screen passes 0 of 5 |
| Temporary accounts, oracle | *Insufficient evidence.* in 5 of 5; without the abstention line, answered in 5 of 5 |


---

## Part F: Save the baseline in `shared/trace.py` (self-study)

Part F keeps the trace step for the labs after this one and saves one traced run of the 30 development questions,
the **baseline**. When a later lab changes a stage, it traces the same questions again and compares them with the
baseline, question by question.

`shared/trace.py` holds Part A's corpus as `corpus()`, Part B's function as `trace()`, and `baseline()`, which traces
the 30 development questions once and saves them in `.cache/`. The file's name holds the chat model and a hash of the
settings, the rules and the questions, as Lab 01's vector cache does, so another model or endpoint gets its own file:


In [21]:
# shared.trace: the module; inspect.getsource() prints its code as written
import inspect
import shared.trace

print(inspect.getsource(shared.trace))


"""Trace records for the pipeline: the steps Lab 04 builds by hand, kept for the labs after it.

trace() runs one question through search, packing and the answer call, and returns what each step produced:
the hits, the packed IDs, a hash of the prompt, the raw reply, the tokens billed and the stamp. baseline() traces
the 30 development questions once and saves them in .cache/, in a file named after the chat model and a hash of
the settings, the rules and the questions, so a changed model or endpoint never reuses old traces.
"""
from __future__ import annotations

import hashlib
import json

from shared.answer import RULES, pack, response_format
from shared.search import chunk_vectors, search
from shared.settings import DATA, ROOT, chat_client, chat_options, stamp


def corpus() -> list[dict]:
    """The 272 chunks plus the five policies, each policy as one chunk whose ID is its source ID."""
    chunks = json.loads((DATA / "squad-chunks.json").read_text(encoding="utf-8"))
    policies 

Run `baseline()` twice: the first call traces the 30 questions and saves them, the second reads the file. Then compare
the prompt hashes with Part B's traces of the same questions, and screen the baseline:


In [22]:
# base: the 30 baseline traces, traced on the first call and read from .cache/ on the second
from shared.trace import baseline

base = baseline(settings)
print(len(base), "traces | read back the same:", baseline(settings) == base)
print("baseline files:", sorted(path.name for path in (ROOT / ".cache").glob("baseline-*.json")))
print("same prompts as Part B:", sum(b["prompt_sha"] == t["prompt_sha"] for b, t in zip(base, traces)), "of", len(base))
print("passed the screen:", sum(screen(parse(b["reply"])["answer"], q["expected"]) for b, q in zip(base, questions)),
      "of", len(base))
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")


30 traces | read back the same: True
baseline files: ['baseline-gpt-4o-mini-c358e23dfda2.json']
same prompts as Part B: 30 of 30
passed the screen: 29 of 30

gpt-4o-mini at https://api.openai.com/v1


30 traces, and the second call read the same records back from the file. All 30 prompt hashes match Part B's: the
same rules met the same evidence, so an answer that differs between the two runs differs because of the model, not
the pipeline. About 29 of 30 pass the screen; the miss is the complexity question. The file's name carries
`gpt-4o-mini`: with another model in `models.env`, `baseline()` traces again into a new file. **From here on the labs
call `baseline()` and `trace()` by name.**

### Checkpoint F

| You should see | Our run |
|---|---|
| The baseline | 30 traces, `read back the same: True`; one file `baseline-gpt-4o-mini-<hash>.json`; same prompts 30 of 30; 29 of 30 pass the screen |


---

## What you can do now

- Make a trace record for one question by hand: hits, packed IDs, prompt hash, raw reply, tokens and stamp.
- Trace a question set with one function and read its cost from the records.
- Screen answers against expected strings by whole words, and read every answer the screen flags.
- Build a stage table and name the stage where each answer lost its support.
- Use oracle context to tell whether the other evidence or the answer step caused a failure, and recognise a right
  answer that an incomplete label calls a miss.
- Save and reload a baseline run with `shared/trace.py`, named by model and endpoint.


## Challenge

Use only what this lab showed. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Find the line that misled the model.** Oracle context said the support alone gives the right answer for the
   mobile-app question. Which of the other packed lines caused the wrong one?

   - Complete `without_each` under **Task 1**: for each packed line of a trace, answer the question three times with
     that line left out, and print the line's ID and the answers.
   - The cell runs it on the mobile-app question and on the complexity question.

   **Check:** for the mobile-app question, leaving out H04 gives *No, you do not need the mobile app ...* in 3 of 3,
   and leaving out any other line keeps a *Yes*. For the complexity question the result moves between runs: in some no single line
   changed the answer; in others leaving out `S04-P00-C00`, the chunk about classifying problems by *inherent
   difficulty*, moved it toward the resources answer (3 of 3 in one run, 1 of 3 in another).

   **Explain:** what repair does each result point to? Why could a line that is true (H04 is right about Boreal) still
   be the cause?

2. **Optional: name the stage in traces that lost the support earlier.** None of the 34 traces lost its support before
   the answer step. Under **Task 2** are three traces written by hand for *How long does an Atlas reset link remain
   valid?*, in the same shape as `trace()`'s records. Run Part D's `stages()` on each and print the stage.

   **Check:** `['retrieval', 'packing', 'generation']`

   **Explain:** for each trace, what would you change first, and which question would you rerun to check it?

**Task 1.** Complete `without_each`, then run the cell:

In [23]:
# without_each(): for each packed line, the three answers given when that line is left out of the evidence
def without_each(question):
    """For each packed line: its ID, and three answers to the question from the other lines."""
    lines = pack(search(question, corpus, vectors, settings, k=5), settings)
    # TODO: for each line, answer the question three times from the other lines (answer(question, shorter, settings));
    #       collect (the left-out line's ID, line[1:line.index("]")], with Counter(...) of the three answers)
    return None

for q in [q for q in questions if "mobile app" in q["question"] or "measure" in q["question"]]:
    result = without_each(q["question"])
    if result is None:
        print("⚠️ Complete without_each, then run the cell again.")
        break
    print(q["question"])
    for left_out, counts in result:
        print(f"  without {left_out:12} {counts.most_common()}")

⚠️ Complete without_each, then run the cell again.


**Task 2 (optional).** Three traces written by hand. Print the stage for each:

In [24]:
# written: three hand-written traces for the Atlas question; stage_of(): the first stage one of them lost the support
atlas = {"question": "How long does an Atlas reset link remain valid?", "expected": ["30 minutes"], "support": ["H02"]}
written = [  # written by hand, not from a run
    {"hits": [["H01", 0.61], ["H04", 0.55], ["H03", 0.40]], "packed": ["H01", "H04"],
     "reply": '{"answer": "Atlas reset links last 24 hours.", "claims": []}'},
    {"hits": [["H01", 0.61], ["H02", 0.58], ["H04", 0.55]], "packed": ["H01"],
     "reply": '{"answer": "Atlas reset links last 24 hours.", "claims": []}'},
    {"hits": [["H02", 0.70], ["H01", 0.61], ["H04", 0.55]], "packed": ["H02", "H01"],
     "reply": '{"answer": "Atlas reset links last 24 hours.", "claims": []}'},
]

def stage_of(t, q):
    # TODO: screen the answer in t["reply"], build the row with stages(t, q, ok),
    #       and return STAGE[...] of the first step that is False
    return None

found = [stage_of(t, atlas) for t in written]
print(found if None not in found else "⚠️ Complete stage_of, then run the cell again.")

⚠️ Complete stage_of, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| The stage table says `support in the hits 4 of 34` | chunk IDs compared with paragraph IDs directly; only the policies match, because a policy's chunk ID is its source ID | Map each chunk ID through `source_of` first, as `stages()` does |
| `KeyError: 'S04-P01-C00'` in Part E | a chunk ID looked up in `passages`, which is keyed by paragraph ID | Use the question's `support` list, or `source_of[chunk_id]` |
| `JSONDecodeError: Expecting value: line 1 column 1 (char 0)` from `parse(t["reply"])` | the reply in that trace is not JSON (a server that ignored `response_format`); the trace keeps the reply as it came | Read the raw `reply` in the record; `answer()` asks once more, `trace()` does not. Try `JSON_MODE=json_object` |

## Resources

- [OpenAI: create chat completion](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create):
  `usage`, the token counts a trace records
- [Python: `hashlib`](https://docs.python.org/3/library/hashlib.html): SHA-256 and `hexdigest()`
- [OpenTelemetry: traces](https://opentelemetry.io/docs/concepts/signals/traces/): how tracing tools record each step
  of a request as a span, the production form of these records
- [Joren et al., *Sufficient Context*, ICLR 2025](https://arxiv.org/abs/2411.06037): separating "the context was not
  enough" from "the model failed with enough context"

Next: [Lab 05](../05_Rebuild_With_LangChain/README.md), *How to rebuild a RAG pipeline with LangChain*. Back to [the lab list](../README.md).